# Demonštračný pipeline pre CH experimenty

Tento notebook slúži ako praktická ukážka pipeline pre segmentáciu koronálnych dier. Kompletný notebook s plnohodnotným tréningom nie je možné jednoducho vložiť do repozitára v spustiteľnej podobe, pretože rozšírená tréningová množina s časovými sekvenciami má veľmi veľký objem. A tréningové dáta preto nie sú súčasťou repozitára.

Pôvodné experimenty boli spúšťané v prostredí Google Colab. Na začiatku sa pripojil Google Drive, potrebné dáta sa skopírovali do lokálneho prostredia Colabu a následne sa spustil celý tréningový pipeline. V tomto repozitári je tréningová časť opísaná slovne podľa pôvodných buniek notebooku. Spustiteľná časť notebooku začína až predikciou na dodatočnej testovacej množine.

Na výber experimentu sa používa konfiguračný súbor z priečinka `configs/`. Podľa zvoleného configu sa načítajú príslušné natrénované modely, dátové archívy a threshold.

---

## Priebeh pôvodného CH pipeline podľa buniek

### Bunka 1: Colab setup, Drive a lokálne kopírovanie dát

V pôvodnom notebooku sa najskôr pripravilo prostredie. Ak sa notebook spúšťal v Google Colab, pripojil sa Google Drive a dáta sa skopírovali z Drive do lokálneho priečinka `/content`. Dôvodom bolo zrýchlenie čítania obrázkov počas tréningu. Pracovalo sa najmä s priečinkami pre CH dáta, časové sekvencie a dodatočnu testovacu množinu.

Schéma pôvodnej logiky:

```python
drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/Experiment")
DATA_ROOT = PROJECT_ROOT / "experiment_data"

# kopírovanie vybraných CH dát do lokálneho pracovného priečinka
LOCAL_DATA_ROOT = Path("/content/ch_data_local")
```

---

### Bunka 2: Importy, parametre experimentu a import projektového kódu

V ďalšej bunke sa importovali knižnice, nastavili sa náhodné seedy, základné parametre modelov a cesty k dátam. Pre CH experiment sa používal kanál 193 Å, veľkosť obrazu 256×256 a sekvencia pozostávajúca z troch predchádzajúcich snímok a cieľového obrazu.

Schéma:

```text
IMG_SIZE = 256
HIST_T = 3
T_STEPS = HIST_T + 1

FILTERS = 32
LAYERS = 4
DROP_PROB = 0.3
CONVLSTM_FILTERS = 32

LOSS_FN = bce_dice_loss
THRESHOLD = 0.5
```

---

### Bunka 3: Vyhľadanie dát priamo z priečinkov

Pôvodný pipeline následne prešiel priečinky s obrázkami a maskami. Pre každý obrázok sa hľadala zodpovedajúca maska a vytváral sa dátový rámec `train_df` a `test_df`.

Dôležité bolo aj spracovanie rotovaných verzií obrázkov. V pôvodnej tréningovej množine boli pre každý nerotovaný obraz prítomné aj verzie otočené o 90, 180 a 270 stupňov. Preto sa pre každý riadok ukladali informácie:

```text
source
image_path
mask_path
stem
base_stem
rot_k
```

Kde `base_stem` označoval pôvodný nerotovaný obraz a `rot_k` určoval rotáciu.

---

### Bunka 4: Kontrola časových sekvencií

Pre model ConvLSTM-SCSS-Net bolo potrebné overiť, či má každý cieľový obraz dostupné tri predchádzajúce snímky:

```text
input_1.png
input_2.png
input_3.png
```

Sekvencie boli pripravené pre nerotované obrázky. Ak bol cieľový obraz rotovaný, historické snímky sa pri načítaní otočili rovnakým uhlom ako cieľový obraz. Tým sa zachovala konzistencia medzi sekvenciou, target obrazom a maskou.

Schéma:

```python
def has_temporal_frames(base_stem):
    seq_dir = FRAMES_DIR / base_stem
    return all((seq_dir / f"input_{i}.png").exists() for i in [1, 2, 3])

train_df = train_df[train_df["base_stem"].apply(has_temporal_frames)]
```

Pre každý záznam sa podľa hodnoty base_stem vyhľadal príslušný priečinok v FRAMES_DIR. Funkcia has_temporal_frames následne skontrolovala, či sa v tomto priečinku nachádzajú všetky tri vstupné snímky. Do tréningovej množiny boli ponechané iba tie záznamy, pre ktoré bola dostupná kompletná časová sekvencia.

---

### Bunka 5: Loadery, augmentácia a generátory

Ďalšia bunka obsahovala najdôležitejšiu časť prípravy dát pre modely. Definovali sa funkcie na načítanie obrazov a masiek, rotácie, načítanie časovej sekvencie a online augmentácia.

Obrázky sa načítavali ako grayscale, menili sa na veľkosť 256×256 a normalizovali sa do rozsahu `[0, 1]`. Masky sa načítavali ako binárne obrazy.

Základný model SCSS-Net dostával jeden obraz:

```text
target image
```

Model ConvLSTM-SCSS-Net dostával sekvenciu:

```text
input_1, input_2, input_3, target image
```

Použitá augmentácia:

```text
horizontal flip
vertical flip
gamma correction
brightness change
```

Pri sekvencii sa rovnaká transformácia aplikovala na všetky snímky aj na masku.

Dôležitou súčasťou tejto bunky bola aj práca s rotovanými verziami obrázkov. Časové sekvencie boli uložené iba pre nerotované obrazy, pretože rotácia nemení časový vzťah medzi snímkami. Ak generátor načítaval rotovaný cieľový obraz, zodpovedajúce snímky sekvencie sa otočili rovnakým uhlom. Rovnaká transformácia sa aplikovala aj na masku. Tým bolo zabezpečené, že vstupná sekvencia, target obraz aj maska zostali geometricky zhodné.

---

### Bunka 6: Dataset a preprocessing sanity checks

Pred tréningom sa kontrolovala štruktúra datasetu. Notebook vypisoval počty obrázkov, počty masiek, rozdelenie dát podľa zdrojov anotácií a počet rotovaných verzií. Zároveň sa kontrolovalo, či generátory vracajú masky v rovnakom poradí ako dátový rámec.

Cieľom bolo overiť, že model nebude trénovaný na nesprávne spárovaných obrázkoch a maskách.

---

### Bunka 7: Vizuálna kontrola batchu

Pred tréningom sa zobrazil jeden batch dát. Pri modeli s časovým kontextom bolo možné vidieť:

```text
input_1
input_2
input_3
target image
mask
```

Táto vizualizácia slúžila na rýchlu kontrolu, že časová sekvencia zodpovedá cieľovému obrazu a že maska patrí k správnemu targetu.

---

### Bunka 8: Callbacks a tréning základného modelu SCSS-Net

Následne sa pripravili výstupné priečinky a callbacks:

```text
ModelCheckpoint
EarlyStopping
ReduceLROnPlateau
CSVLogger
```

Potom sa zostavil a trénoval základný model SCSS-Net. Tento model pracoval iba s jedným cieľovým obrazom.

Schéma:

```python
baseline_model = scss_net(...)
baseline_model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss=bce_dice_loss,
    metrics=[dice_soft, iou_soft]
)

history_base = baseline_model.fit(
    train_base_gen,
    validation_data=val_base_gen,
    epochs=EPOCHS,
    callbacks=callbacks
)
```

---

### Bunka 9: Tréning modelu ConvLSTM-SCSS-Net

Druhá trénovacia bunka vytvárala model s časovým kontextom. Na vstupe mal sekvenciu tvaru:

```text
(4, 256, 256, 1)
```

Prvé tri snímky predstavovali časový kontext a posledná snímka bola target image. Model mal predikovať masku pre target obraz.

Schéma:

```python
temporal_model = scss_net_convlstm_early(
    input_shape=(4, 256, 256, 1),
    filters=FILTERS,
    layers=LAYERS,
    drop_prob=DROP_PROB,
    convlstm_filters=CONVLSTM_FILTERS
)

history_temp = temporal_model.fit(
    train_temp_gen,
    validation_data=val_temp_gen,
    epochs=EPOCHS,
    callbacks=callbacks
)
```

---

### Bunka 10: Training curves

Po tréningu sa vykresľovali grafy priebehu tréningu. Porovnávali sa hodnoty loss, Dice a IoU pre tréningovú a validačnú časť. Tieto grafy slúžili na kontrolu stability učenia a možného preučenia modelu.

![Training curves](figures/CH_training_curves_comparison.png)

---

### Bunka 11: Vyhodnotenie na pôvodnom testovacom splite

Po trénovaní sa modely vyhodnotili na pôvodnom testovacom splite. Vypočítali sa Dice a IoU a vytvorili sa testovacie vizualizácie. Táto časť slúžila na kontrolu výsledkov na pôvodnej testovacej množine pred prechodom na dodatočnú testovaciu množinu.

![Test split preview](figures/CH_test_preview_5_examples.png)

---

## Predikcia na dodatočnej testovacej množine

Po vyhodnotení modelov na pôvodnom testovacom splite nasledovala predikcia na dodatočnej testovacej množine. V štandardnej konfigurácii išlo o množinu z roku 2021, ktorá nebola použitá pri trénovaní modelov. Slúžila na následné porovnanie správania základného modelu SCSS-Net a modelu ConvLSTM-SCSS-Net na samostatných dátach. Práve predikcie vytvorené na tejto množine boli použité pri vizuálnej interpretácii výsledkov experimentov.

V pôvodnom notebooku sa v tejto časti načítali obrazy, masky a časové sekvencie z dodatočnej testovacej množiny, následne sa načítali už natrénované modely a vykonala sa predikcia oboma modelmi. Výstupom boli pravdepodobnostné mapy, binárne masky, metriky a koláže s vizuálnym porovnaním oboch modelov.

Dodatočné testovacie množiny sú výrazne menšie než kompletný tréningový dataset, preto ich bolo možné zahrnúť do repozitára. Súčasťou repozitára sú aj už natrénované modely pre jednotlivé experimentálne konfigurácie. V tejto demonštračnej verzii preto používateľ najskôr zvolí konfiguračný súbor. Podľa neho sa automaticky načíta dvojica modelov, teda základný model SCSS-Net a model ConvLSTM-SCSS-Net, rozbalia sa príslušné dáta a následne sa vykoná predikcia rovnakým spôsobom ako v pôvodnom pipeline.

Výstupy sa ukladajú do priečinka definovaného v konfigurácii. Ukladajú sa binárne masky oboch modelov, pravdepodobnostné mapy, CSV súbor s metrikami, koláže s vizuálnym porovnaním a použitý konfiguračný súbor.


In [ ]:
# =========================
# IMPORTY A VÝBER CONFIGU PRE PREDIKCIU
# =========================
from pathlib import Path
import sys
import json
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf

# Dostupné CH konfigurácie:
#   configs/CH_standard.json
#   configs/CH_low.json
#   configs/CH_region_growth.json
#   configs/CH_2025_raw.json
#   configs/CH_2025_processed.json

CONFIG_PATH = "configs/CH_standard.json"

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

CONFIG_FILE = PROJECT_ROOT / CONFIG_PATH
if not CONFIG_FILE.exists():
    raise FileNotFoundError(f"Konfiguračný súbor sa nenašiel: {CONFIG_FILE}")

with open(CONFIG_FILE, "r", encoding="utf-8") as f:
    CONFIG = json.load(f)

TRAINING_CFG = CONFIG["training_config"]
PRED_CFG = CONFIG["prediction_config"]

SRC_DIR = PROJECT_ROOT / "src"
sys.path.insert(0, str(SRC_DIR))

from metrics import dice_np, iou_np, dice_soft, iou_soft, bce_dice_loss

IMG_SIZE = int(TRAINING_CFG["image_size"][0])
HIST_T = int(TRAINING_CFG["previous_frames"])
T_STEPS = int(TRAINING_CFG["sequence_length"])
USE_CURRENT_FRAME = bool(TRAINING_CFG.get("target_frame_included", True))
THRESHOLD = float(PRED_CFG["threshold"])

BASELINE_MODEL_PATH = PROJECT_ROOT / PRED_CFG["models"]["baseline"]
CONVLSTM_MODEL_PATH = PROJECT_ROOT / PRED_CFG["models"]["convlstm"]

OUT_DIR = PROJECT_ROOT / PRED_CFG["output_dir"]
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Experiment:", CONFIG["experiment_id"])
print("Popis:", CONFIG.get("description", ""))
print("Threshold:", THRESHOLD)
print("Baseline model:", BASELINE_MODEL_PATH)
print("ConvLSTM model:", CONVLSTM_MODEL_PATH)
print("Archívy:")
for archive in PRED_CFG["archives"]:
    print(" -", PROJECT_ROOT / archive)
print("Výstupy:", OUT_DIR)

In [ ]:
# =========================
# PREDIKCIA PODĽA ZVOLENÉHO CONFIGU
# =========================

IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

# None = spracovať celú dodatočnú testovaciu množinu.
# Napr. 5 alebo 20 = rýchla lokálna kontrola.
MAX_SAMPLES = None

def list_image_files(root: Path):
    return sorted([
        p for p in root.rglob("*")
        if p.is_file() and p.suffix.lower() in IMG_EXTS
    ])

def clean_key(path: Path):
    key = path.stem.lower()
    for token in ["_mask", "_masks", "_image", "_target", "mask_", "image_", "target_"]:
        key = key.replace(token, "")
    return key

def unzip_archives(archives, extract_root: Path):
    extract_root.mkdir(parents=True, exist_ok=True)

    for rel_path in archives:
        archive_path = PROJECT_ROOT / rel_path
        if not archive_path.exists():
            raise FileNotFoundError(f"Archív sa nenašiel: {archive_path}")

        target_dir = extract_root / archive_path.stem
        target_dir.mkdir(parents=True, exist_ok=True)

        existing_files = [p for p in target_dir.rglob("*") if p.is_file()]
        if existing_files:
            print(f"Už rozbalené: {target_dir} ({len(existing_files)} súborov)")
            continue

        print(f"Rozbaľujem {archive_path.name} -> {target_dir}")
        with zipfile.ZipFile(archive_path, "r") as zf:
            zf.extractall(target_dir)

def path_contains(path: Path, keywords):
    text = "/".join(str(x).lower() for x in path.parts)
    return any(k in text for k in keywords)

def find_input_frame(seq_dir: Path, idx: int):
    for ext in IMG_EXTS:
        candidate = seq_dir / f"input_{idx}{ext}"
        if candidate.exists():
            return candidate
    return None

def find_sequence_folders(root: Path):
    folders = []

    for folder in root.rglob("*"):
        if not folder.is_dir():
            continue

        ok = True
        for i in range(1, HIST_T + 1):
            if find_input_frame(folder, i) is None:
                ok = False
                break

        if ok:
            folders.append(folder)

    return sorted(folders)

def is_inside_any(path: Path, folders):
    return any(folder in path.parents for folder in folders)

def load_gray_float(path: Path):
    img = Image.open(path).convert("L").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
    arr = np.asarray(img, dtype=np.float32) / 255.0
    return arr[..., None]

def load_mask_float(path: Path):
    img = Image.open(path).convert("L").resize((IMG_SIZE, IMG_SIZE), Image.NEAREST)
    arr = (np.asarray(img, dtype=np.float32) > 127).astype(np.float32)
    return arr[..., None]

def load_sequence(seq_dir, target_path: Path):
    target = load_gray_float(target_path)

    if seq_dir is None:
        return np.stack([target] * T_STEPS, axis=0)

    frames = []

    for i in range(1, HIST_T + 1):
        frame_path = find_input_frame(Path(seq_dir), i)
        frames.append(load_gray_float(frame_path) if frame_path is not None else target)

    if USE_CURRENT_FRAME:
        frames.append(target)

    while len(frames) < T_STEPS:
        frames.append(target)

    return np.stack(frames[:T_STEPS], axis=0)

def find_prediction_samples(extract_root: Path):
    all_files = list_image_files(extract_root)
    sequence_folders = find_sequence_folders(extract_root)

    mask_files = [p for p in all_files if path_contains(p, ["mask", "masks"])]
    sequence_files = [p for p in all_files if is_inside_any(p, sequence_folders)]

    image_files = [
        p for p in all_files
        if p not in mask_files
        and p not in sequence_files
        and not path_contains(p, ["mask", "masks", "sequence", "sequences", "seq"])
    ]

    image_map = {clean_key(p): p for p in image_files}
    mask_map = {clean_key(p): p for p in mask_files}
    seq_map = {clean_key(p): p for p in sequence_folders}

    common = sorted(set(image_map) & set(mask_map))

    rows = []

    if common:
        for idx, key in enumerate(common):
            seq_dir = seq_map.get(key)
            if seq_dir is None and idx < len(sequence_folders):
                seq_dir = sequence_folders[idx]

            rows.append({
                "sample_id": key,
                "image_path": image_map[key],
                "mask_path": mask_map[key],
                "sequence_dir": seq_dir
            })
    else:
        n = min(len(image_files), len(mask_files))
        for idx in range(n):
            rows.append({
                "sample_id": f"sample_{idx:04d}",
                "image_path": image_files[idx],
                "mask_path": mask_files[idx],
                "sequence_dir": sequence_folders[idx] if idx < len(sequence_folders) else None
            })

    return pd.DataFrame(rows), image_files, mask_files, sequence_folders

def model_to_model_iou(a, b):
    a = a.astype(bool)
    b = b.astype(bool)
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float((inter + 1e-7) / (union + 1e-7))

def difference_rgb(base_mask, conv_mask):
    base = np.squeeze(base_mask).astype(bool)
    conv = np.squeeze(conv_mask).astype(bool)

    rgb = np.zeros((base.shape[0], base.shape[1], 3), dtype=np.uint8)
    rgb[base & ~conv] = [255, 0, 0]
    rgb[conv & ~base] = [0, 180, 255]
    rgb[base & conv] = [255, 255, 0]
    return rgb

def require_file(path: Path, label: str):
    if not path.exists():
        raise FileNotFoundError(f"{label} sa nenašiel: {path}")

# 1. Rozbalenie dát
EXTRACT_DIR = OUT_DIR / "extracted_prediction_data"
unzip_archives(PRED_CFG["archives"], EXTRACT_DIR)

final_df, found_images, found_masks, found_sequences = find_prediction_samples(EXTRACT_DIR)

print("Nájdené obrazy:", len(found_images))
print("Nájdené masky:", len(found_masks))
print("Nájdené sekvencie:", len(found_sequences))
print("Spárované vzorky:", len(final_df))

if len(final_df) == 0:
    raise RuntimeError("Nepodarilo sa vytvoriť final_df. Skontrolujte štruktúru archívov.")

display(final_df.head())

if MAX_SAMPLES is not None:
    final_df = final_df.head(MAX_SAMPLES).copy()
    print(f"Demo režim: spracuje sa iba prvých {len(final_df)} vzoriek.")

# 2. Načítanie modelov
require_file(BASELINE_MODEL_PATH, "Baseline model")
require_file(CONVLSTM_MODEL_PATH, "ConvLSTM model")

custom_objects = {
    "dice_soft": dice_soft,
    "iou_soft": iou_soft,
    "bce_dice_loss": bce_dice_loss
}

baseline_model = tf.keras.models.load_model(BASELINE_MODEL_PATH, custom_objects=custom_objects)
convlstm_model = tf.keras.models.load_model(CONVLSTM_MODEL_PATH, custom_objects=custom_objects)

print("Načítaný baseline model:", BASELINE_MODEL_PATH)
print("Načítaný ConvLSTM model:", CONVLSTM_MODEL_PATH)

# 3. Priečinky pre výstupy
PRED_DIR = OUT_DIR / "prediction_on_additional_evaluation_set"
BASE_MASK_DIR = PRED_DIR / "baseline_masks"
CONV_MASK_DIR = PRED_DIR / "convlstm_masks"
COLLAGE_DIR = PRED_DIR / "collages"

for folder in [BASE_MASK_DIR, CONV_MASK_DIR, COLLAGE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# 4. Predikcia po jednom príklade
metrics_rows = []

for i, row in final_df.reset_index(drop=True).iterrows():
    sample_id = str(row["sample_id"])
    print(f"[{i + 1}/{len(final_df)}] Predikcia: {sample_id}")

    target = load_gray_float(row["image_path"])
    mask = load_mask_float(row["mask_path"])
    seq = load_sequence(row["sequence_dir"], row["image_path"])

    x_base = target[None, ...]
    x_temp = seq[None, ...]

    baseline_prob_one = baseline_model.predict(x_base, batch_size=1, verbose=0)[0, ..., 0]
    convlstm_prob_one = convlstm_model.predict(x_temp, batch_size=1, verbose=0)[0, ..., 0]

    baseline_pred_one = (baseline_prob_one > THRESHOLD).astype(np.float32)
    convlstm_pred_one = (convlstm_prob_one > THRESHOLD).astype(np.float32)

    y_true_one = mask[..., 0]

    Image.fromarray((baseline_pred_one * 255).astype(np.uint8)).save(
        BASE_MASK_DIR / f"{sample_id}_baseline.png"
    )
    Image.fromarray((convlstm_pred_one * 255).astype(np.uint8)).save(
        CONV_MASK_DIR / f"{sample_id}_convlstm.png"
    )

    metrics_rows.append({
        "sample_id": sample_id,
        "baseline_dice": dice_np(y_true_one, baseline_pred_one),
        "baseline_iou": iou_np(y_true_one, baseline_pred_one),
        "convlstm_dice": dice_np(y_true_one, convlstm_pred_one),
        "convlstm_iou": iou_np(y_true_one, convlstm_pred_one),
        "baseline_area_ratio": float(baseline_pred_one.mean()),
        "convlstm_area_ratio": float(convlstm_pred_one.mean()),
        "delta_area_convlstm_minus_baseline": float(convlstm_pred_one.mean() - baseline_pred_one.mean()),
        "iou_between_models": model_to_model_iou(baseline_pred_one, convlstm_pred_one),
        "xor_ratio": float(np.logical_xor(
            baseline_pred_one.astype(bool),
            convlstm_pred_one.astype(bool)
        ).mean())
    })

    # Koláž sa ukladá pre každý spracovaný príklad.
    fig, axes = plt.subplots(1, 5, figsize=(18, 4))

    axes[0].imshow(target[..., 0], cmap="gray")
    axes[0].set_title("Image")

    axes[1].imshow(y_true_one, cmap="gray")
    axes[1].set_title("True mask")

    axes[2].imshow(baseline_pred_one, cmap="gray")
    axes[2].set_title("Baseline pred")

    axes[3].imshow(convlstm_pred_one, cmap="gray")
    axes[3].set_title("ConvLSTM pred")

    axes[4].imshow(target[..., 0], cmap="gray")
    axes[4].imshow(difference_rgb(baseline_pred_one, convlstm_pred_one), alpha=0.55)
    axes[4].set_title("Overlay comparison")

    for ax in axes:
        ax.axis("off")

    plt.suptitle(sample_id)
    plt.tight_layout()
    plt.savefig(COLLAGE_DIR / f"{sample_id}_comparison.png", dpi=160, bbox_inches="tight")
    plt.close(fig)

# 5. Uloženie metrík a configu
metrics_df = pd.DataFrame(metrics_rows)

metrics_path = PRED_DIR / f"{CONFIG['experiment_id']}_metrics.csv"
metrics_df.to_csv(metrics_path, index=False)

with open(PRED_DIR / "used_config.json", "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2, ensure_ascii=False)

print("Hotovo.")
print("Výstupy:", PRED_DIR)
print("Metriky:", metrics_path)
print("Koláže:", COLLAGE_DIR)

display(metrics_df.head())
display(metrics_df.describe())

# 6. Zobrazenie niekoľkých uložených koláží v notebooku
preview_collages = sorted(COLLAGE_DIR.glob("*_comparison.png"))[:3]

if preview_collages:
    for collage_path in preview_collages:
        img = Image.open(collage_path)

        plt.figure(figsize=(18, 4))
        plt.imshow(img)
        plt.axis("off")
        plt.title(collage_path.name)
        plt.show()
else:
    print("Neboli nájdené žiadne uložené koláže na zobrazenie.")